# Agent y System: seis ejemplos, una gramática

**Secuencia, paralelo y decisiones**, ejecutados con **native y LangGraph**.
Cada ejemplo muestra **input → proceso observado → output** y comprueba la salida.
Después conectamos las mismas unidades a **Skills, Environment y Eval**, sin
añadir constructores ni una segunda forma de ejecutarlas.

Este es un prototipo aislado (`unified_grammar`), no una ampliación publicada de
Agentic Systems 2.1.2. Sus Tools son operaciones Python deterministas: no se llama
a ningún modelo, proveedor, Ollama ni servicio AWS. Las dos implementaciones del
flujo son reales; no se simula LangGraph.

Ejecuta las celdas en orden en un **kernel dedicado**. Instala antes
`requirements.txt` y Jupyter/ipykernel. La primera celda activa un bloqueo de
conexiones de aplicación y subprocesos que dura hasta cerrar este kernel.

In [1]:
from pathlib import Path
import sys
import json
import html
import importlib.metadata as metadata
from IPython.display import HTML, display

candidates = [
    candidate
    for parent in (Path.cwd(), *Path.cwd().parents)
    for candidate in (parent / "examples", parent)
]
source_root = next(
    p for p in candidates if (p / "unified_grammar" / "core.py").is_file()
)
sys.path.insert(0, str(source_root))

from unified_grammar.offline import enforce_offline, blocked_attempts
enforce_offline()

import agentic_systems as toolkit
from unified_grammar import Action, Agent, System, Sequence, Parallel, Graph, Environment, Eval

assert metadata.version("agentic-systems") == toolkit.__version__ == "2.1.2"
assert "site-packages" in Path(toolkit.__file__).parts, "Usa el wheel sin modo editable"
BACKENDS = ("native", "langgraph")
print({
    "Python": sys.version.split()[0],
    "agentic-systems": toolkit.__version__,
    "LangGraph": metadata.version("langgraph"),
    "inferencia": "deshabilitada",
})

{'Python': '3.14.2', 'agentic-systems': '2.1.2', 'LangGraph': '1.2.16', 'inferencia': 'deshabilitada'}


## Dos Tools y una Skill

Las Tools calculan; la Skill agrupa sus capacidades. `Action` coloca una Tool
en un paso del flujo. El Agent tiene disponibles sus propias Skills.
Estas funciones proceden del mismo `demo.py` usado por el CLI y las pruebas.

`Action("increase", "increment")` busca la Tool llamada `increment` en las
Skills de su Agent. El nombre del paso (`increase`) puede ser diferente del
nombre de la Tool. La Skill no tiene una ejecución separada: se conecta con
`skills=(arithmetic,)` y el Agent ejecuta las Tools mediante su flujo.

In [2]:
@toolkit.tool
def increment(value: int) -> dict:
    """Increment an integer."""
    return {"value": value + 1}

@toolkit.tool
def double(value: int) -> dict:
    """Double an integer."""
    return {"value": value * 2}

arithmetic = toolkit.skill(name="arithmetic", tools=[increment, double])
increase = Action("increase", "increment")
multiply = Action("multiply", "double")

## Cómo mostramos el recorrido

La siguiente función es solo presentación y comprobación del notebook; no es
otro constructor de la librería. Ejecuta `unit.run(input)`, lee los pasos y
destinos conservados en `RunResult`, y comprueba output, serialización y linaje.
En paralelo se muestran las ramas en orden declarado, no como una cronología.

In [3]:
records = []
prepared_units = []

def render_io(backend, payload, process, output, status="PASS"):
    values = [backend, payload, process, output, status]
    headers = ["Backend", "Input", "Proceso observado", "Output", "Estado"]
    display(HTML("<table><tr>" + "".join(f"<th>{h}</th>" for h in headers)
                 + "</tr><tr>" + "".join(
                     f"<td><code>{html.escape(str(v))}</code></td>" for v in values
                 ) + "</tr></table>"))

def observed_process(result):
    steps = [f"{n.meta['name']}: {n.data}" for n in result.walk()
             if n.meta.get("prototype_kind") == "action"]
    routes = [f"{n.meta['graph_node']} → {n.meta['next']}" for n in result.walk()
              if "graph_node" in n.meta and "next" in n.meta]
    return "; ".join(steps + routes)

def show(owner, flow, backend, unit, payload, expected, case="principal"):
    before = json.loads(json.dumps(payload))
    result = unit.run(payload)
    assert result.ok, result.to_dict()
    assert result.data == expected, result.to_dict()
    assert payload == before
    nodes = list(result.walk())
    assert len({n.execution_id for n in nodes}) == len(nodes)
    for node in nodes:
        for child in node.children:
            assert child.parent_execution_id == node.execution_id
    steps = [
        {"step": n.meta["name"], "output": n.data}
        for n in nodes if n.meta.get("prototype_kind") == "action"
    ]
    routes = [
        {"from": n.meta["graph_node"], "to": n.meta["next"]}
        for n in nodes if "graph_node" in n.meta and "next" in n.meta
    ]
    expected_steps = 1 if flow == "graph" and payload["value"] + 1 < 0 else 2
    events = [e for n in nodes for e in n.tool_events]
    assert len(steps) == len(events) == expected_steps
    if flow == "graph":
        assert routes[0]["to"] == ("multiply" if payload["value"] + 1 >= 0 else "END")
    if owner == "System":
        agents = [n.meta["name"] for n in nodes
                  if n.meta.get("prototype_kind") == "agent"]
        assert agents == (["incrementer"] if expected_steps == 1
                          else ["incrementer", "doubler"])
    separator = " | " if flow == "parallel" else " → "
    process = separator.join(f"{s['step']}: {s['output']}" for s in steps)
    if flow == "parallel":
        process += " → sumar salidas de ambas ramas"
    if routes:
        process += " · ruta: " + ", ".join(f"{r['from']} → {r['to']}" for r in routes)
    row = dict(owner=owner, flow=flow, backend=backend, case=case,
               input=before, process=process, steps=steps, routes=routes,
               output=result.data, passed=True, result=result.to_dict())
    json.dumps(row, allow_nan=False)
    records.append(row)
    prepared_units.append((unit, before, expected))
    render_io(backend, before, process, result.data)

## 1. Agent en secuencia

Cada paso recibe la salida del anterior: **3 → incrementar: 4 → duplicar: 8**.

In [4]:
for backend in BACKENDS:
    agent_sequence = Agent(
        name="agent_sequential",
        skills=(arithmetic,),
        flow=Sequence((increase, multiply)),
        backend=backend,
    )
    show("Agent", "sequential", backend, agent_sequence,
         {"value": 3}, {"value": 8})

Backend,Input,Proceso observado,Output,Estado
native,{'value': 3},increase: {'value': 4} → multiply: {'value': 8},{'value': 8},PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,{'value': 3},increase: {'value': 4} → multiply: {'value': 8},{'value': 8},PASS


## 2. Agent en paralelo

Las dos ramas reciben **3** de forma independiente: incrementar da **4**,
duplicar da **6**. `join` suma sus salidas: **10**.

In [5]:
def add_branches(outputs):
    return {"value": sum(child["value"] for child in outputs.values())}

for backend in BACKENDS:
    agent_parallel = Agent(
        name="agent_parallel",
        skills=(arithmetic,),
        flow=Parallel((increase, multiply), join=add_branches),
        backend=backend,
    )
    show("Agent", "parallel", backend, agent_parallel,
         {"value": 3}, {"value": 10})

Backend,Input,Proceso observado,Output,Estado
native,{'value': 3},increase: {'value': 4} | multiply: {'value': 6} → sumar salidas de ambas ramas,{'value': 10},PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,{'value': 3},increase: {'value': 4} | multiply: {'value': 6} → sumar salidas de ambas ramas,{'value': 10},PASS


## 3. Agent con decisión

La conexión recibe una **función Python directa**. No hay `Route` ni lista de
destinos repetida. Primero se incrementa; la función examina ese resultado y
devuelve el siguiente nodo o `"END"`.

- **3 → 4 → duplicar → 8**.
- **−3 → −2 → terminar → −2**: comprobamos también la rama alternativa.

El grafo admite como destinos sus nodos y `END`; valida lo elegido al ejecutar.
La compilación no ejecuta la condición ni intenta deducirla leyendo su código.

In [6]:
def choose_next(data):
    if data["value"] >= 0:
        return "multiply"
    return "END"

for backend in BACKENDS:
    agent_decision = Agent(
        name="agent_graph",
        skills=(arithmetic,),
        flow=Graph(
            start="increase",
            nodes=(("increase", increase), ("multiply", multiply)),
            edges=(("increase", choose_next), ("multiply", "END")),
        ),
        backend=backend,
    )
    show("Agent", "graph", backend, agent_decision,
         {"value": 3}, {"value": 8})
    show("Agent", "graph", backend, agent_decision,
         {"value": -3}, {"value": -2}, case="alternativa")

Backend,Input,Proceso observado,Output,Estado
native,{'value': 3},"increase: {'value': 4} → multiply: {'value': 8} · ruta: increase → multiply, multiply → END",{'value': 8},PASS


Backend,Input,Proceso observado,Output,Estado
native,{'value': -3},increase: {'value': -2} · ruta: increase → END,{'value': -2},PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,{'value': 3},"increase: {'value': 4} → multiply: {'value': 8} · ruta: increase → multiply, multiply → END",{'value': 8},PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,{'value': -3},increase: {'value': -2} · ruta: increase → END,{'value': -2},PASS


## Ahora coordinamos dos agentes

El System usará las mismas composiciones. Sus pasos son Agents con identidad y
Skills propias. Preparamos un par por backend; el backend del System no cambia
automáticamente el de sus hijos.

In [7]:
workers = {}
for backend in BACKENDS:
    incrementer = Agent(
        name="incrementer", flow=increase,
        skills=(arithmetic,), backend=backend,
    )
    doubler = Agent(
        name="doubler", flow=multiply,
        skills=(arithmetic,), backend=backend,
    )
    workers[backend] = (incrementer, doubler)

## 4. System en secuencia

**3 → incrementer: 4 → doubler: 8**. Cambia el propietario de los pasos,
no la gramática de la secuencia.

In [8]:
for backend in BACKENDS:
    incrementer, doubler = workers[backend]
    system_sequence = System(
        name="system_sequential",
        flow=Sequence((incrementer, doubler)),
        backend=backend,
    )
    show("System", "sequential", backend, system_sequence,
         {"value": 3}, {"value": 8})

Backend,Input,Proceso observado,Output,Estado
native,{'value': 3},increase: {'value': 4} → multiply: {'value': 8},{'value': 8},PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,{'value': 3},increase: {'value': 4} → multiply: {'value': 8},{'value': 8},PASS


## 5. System en paralelo

Los dos Agents reciben **3**. Sus salidas **4** y **6** se reúnen con la misma
función `add_branches`: **10**. El resultado conserva ambos Agents como hijos.

In [9]:
for backend in BACKENDS:
    incrementer, doubler = workers[backend]
    system_parallel = System(
        name="system_parallel",
        flow=Parallel((incrementer, doubler), join=add_branches),
        backend=backend,
    )
    show("System", "parallel", backend, system_parallel,
         {"value": 3}, {"value": 10})

Backend,Input,Proceso observado,Output,Estado
native,{'value': 3},increase: {'value': 4} | multiply: {'value': 6} → sumar salidas de ambas ramas,{'value': 10},PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,{'value': 3},increase: {'value': 4} | multiply: {'value': 6} → sumar salidas de ambas ramas,{'value': 10},PASS


## 6. System con decisión

El mismo `choose_next` decide qué Agent ejecutar después. Las dos rutas producen
**8** y **−2**, igual que en el Agent con decisión.

In [10]:
for backend in BACKENDS:
    incrementer, doubler = workers[backend]
    system_decision = System(
        name="system_graph",
        flow=Graph(
            start="increase",
            nodes=(("increase", incrementer), ("multiply", doubler)),
            edges=(("increase", choose_next), ("multiply", "END")),
        ),
        backend=backend,
    )
    show("System", "graph", backend, system_decision,
         {"value": 3}, {"value": 8})
    show("System", "graph", backend, system_decision,
         {"value": -3}, {"value": -2}, case="alternativa")

Backend,Input,Proceso observado,Output,Estado
native,{'value': 3},"increase: {'value': 4} → multiply: {'value': 8} · ruta: increase → multiply, multiply → END",{'value': 8},PASS


Backend,Input,Proceso observado,Output,Estado
native,{'value': -3},increase: {'value': -2} · ruta: increase → END,{'value': -2},PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,{'value': 3},"increase: {'value': 4} → multiply: {'value': 8} · ruta: increase → multiply, multiply → END",{'value': 8},PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,{'value': -3},increase: {'value': -2} · ruta: increase → END,{'value': -2},PASS


## Resultado de los seis ejemplos

La matriz principal tiene **6 ejemplos × 2 backends = 12 combinaciones**.
Las dos decisiones añaden **4 comprobaciones de la rama alternativa**.
Los siguientes datos se calculan a partir de los resultados anteriores.

In [11]:
principal = [r for r in records if r["case"] == "principal"]
alternatives = [r for r in records if r["case"] == "alternativa"]
assert len(principal) == 12 and len(alternatives) == 4
for owner in ("Agent", "System"):
    for flow in ("sequential", "parallel", "graph"):
        pair = [r for r in principal if r["owner"] == owner and r["flow"] == flow]
        assert {r["backend"] for r in pair} == set(BACKENDS)
        assert pair[0]["output"] == pair[1]["output"]
headers = ["Owner", "Flujo", "Backend", "Input", "Output", "Estado"]
table = "<table><tr>" + "".join(f"<th>{h}</th>" for h in headers) + "</tr>"
for row in principal:
    values = [row["owner"], row["flow"], row["backend"], row["input"], row["output"], "PASS"]
    table += "<tr>" + "".join(f"<td>{html.escape(str(v))}</td>" for v in values) + "</tr>"
display(HTML(table + "</table>"))
print(f"Matriz: {len(principal)}/12; alternativas: {len(alternatives)}/4")

Owner,Flujo,Backend,Input,Output,Estado
Agent,sequential,native,{'value': 3},{'value': 8},PASS
Agent,sequential,langgraph,{'value': 3},{'value': 8},PASS
Agent,parallel,native,{'value': 3},{'value': 10},PASS
Agent,parallel,langgraph,{'value': 3},{'value': 10},PASS
Agent,graph,native,{'value': 3},{'value': 8},PASS
Agent,graph,langgraph,{'value': 3},{'value': 8},PASS
System,sequential,native,{'value': 3},{'value': 8},PASS
System,sequential,langgraph,{'value': 3},{'value': 8},PASS
System,parallel,native,{'value': 3},{'value': 10},PASS
System,parallel,langgraph,{'value': 3},{'value': 10},PASS


Matriz: 12/12; alternativas: 4/4


## 7. Skills: las capacidades pertenecen a cada Agent

Ya conectamos `arithmetic` mediante `skills=(arithmetic,)`. El recorrido es:

**input → Agent → Action → Tool resuelta en su Skill → output**.

En un System, cada Agent conserva su Skill. El System coordina esos agentes;
no necesita copiar sus capacidades. Aquí recuperamos las mismas unidades y
mostramos los resultados ya obtenidos, sin reconstruir otra aplicación.

In [12]:
examples = {
    (row["owner"], row["flow"], row["backend"]): unit
    for (unit, _, _), row in zip(prepared_units, records)
    if row["case"] == "principal"
}
assert len(examples) == 12
skill_connections = []
for backend in BACKENDS:
    agent = examples[("Agent", "sequential", backend)]
    application = examples[("System", "graph", backend)]
    assert agent.skills == (arithmetic,)
    assert application.skills == ()
    assert all(worker.skills == (arithmetic,) for worker in workers[backend])
    tools = [tool.name for tool in arithmetic.tools]
    assert tools == ["increment", "double"]
    row = next(r for r in principal if r["owner"] == "Agent"
               and r["flow"] == "sequential" and r["backend"] == backend)
    assert row["result"]["meta"]["skills"] == ["arithmetic"]
    render_io(backend, row["input"],
              f"Agent.skills → arithmetic → {tools}; {row['process']}", row["output"])
    skill_connections.append({
        "backend": backend, "skill": arithmetic.name, "tools": tools,
        "agents_in_system": [worker.name for worker in workers[backend]],
        "input": row["input"], "process": row["process"], "output": row["output"],
    })

Backend,Input,Proceso observado,Output,Estado
native,{'value': 3},"Agent.skills → arithmetic → ['increment', 'double']; increase: {'value': 4} → multiply: {'value': 8}",{'value': 8},PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,{'value': 3},"Agent.skills → arithmetic → ['increment', 'double']; increase: {'value': 4} → multiply: {'value': 8}",{'value': 8},PASS


## 8. Environment: varios inputs, una misma unidad

Environment entrega un registro por paso al **System con decisión del ejemplo 6**.
El backend y las Skills siguen perteneciendo a la unidad que ya construimos.

```python
episode = Environment(name="two_steps", unit=application, records=inputs)
episode_result = episode.run()
```

**Input:** dos registros, `3` y `−3`.
**Proceso:** abrir un episodio, ejecutar el System por cada registro y conservar
transiciones y memoria del episodio.
**Output:** `8` y `−2`, dos pasos y memoria `{"processed": 2}`.

Este ejemplo alimenta inputs independientes; la memoria del episodio no se
inyecta automáticamente en el input de los Agents. Environment utiliza las
sesiones reales de 2.1.2 mediante el wrapper experimental.

In [13]:
inputs = ({"value": 3}, {"value": -3})
environment_examples = []
for backend in BACKENDS:
    application = examples[("System", "graph", backend)]
    episode = Environment(name="two_steps", unit=application, records=inputs)
    episode_result = episode.run()

    assert episode_result.ok
    assert episode_result.data["outputs"] == [{"value": 8}, {"value": -2}]
    assert episode_result.data["summary"]["memory"] == {"processed": 2}
    assert len(episode_result.data["history"]) == len(episode_result.children) == 2
    for transition, child in zip(episode_result.data["history"], episode_result.children):
        assert transition["graph_state"]["run"]["execution_id"] == child.execution_id
        render_io(backend, transition["row"], observed_process(child), child.data)
    summary = episode_result.data["summary"]
    print(backend, "episodio:", {key: summary[key] for key in ("steps", "done", "memory")})
    environment_examples.append({
        "backend": backend, "input": list(inputs),
        "output": episode_result.data["outputs"], "result": episode_result.to_dict(),
    })

Backend,Input,Proceso observado,Output,Estado
native,{'value': 3},increase: {'value': 4}; multiply: {'value': 8}; increase → multiply; multiply → END,{'value': 8},PASS


Backend,Input,Proceso observado,Output,Estado
native,{'value': -3},increase: {'value': -2}; increase → END,{'value': -2},PASS


native episodio: {'steps': 2, 'done': True, 'memory': {'processed': 2}}


Backend,Input,Proceso observado,Output,Estado
langgraph,{'value': 3},increase: {'value': 4}; multiply: {'value': 8}; increase → multiply; multiply → END,{'value': 8},PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,{'value': -3},increase: {'value': -2}; increase → END,{'value': -2},PASS


langgraph episodio: {'steps': 2, 'done': True, 'memory': {'processed': 2}}


## 9. Eval: inputs y expectativas para la misma unidad

Eval ejecuta el **mismo System**, compara sus resultados y conserva un informe.
No se conecta a Environment en este prototipo: ambos reciben un Agent o System.

```python
evaluation = Eval(name="decision_contract", unit=application, cases=cases)
evaluation_result = evaluation.run()
```

**Input:** dos casos con input y salida esperada.
**Proceso:** ejecutar la unidad y comprobar `ok` y `data` exactos, incluidos sus tipos.
**Output:** dos casos aprobados y cero fallos por backend. Es una evaluación
determinista; no hay juez LLM. Los casos evalúan el resultado final; el linaje y
los eventos se comprueban además en la sección de verificación.

In [14]:
cases = (
    {"name": "positive", "input": {"value": 3},
     "expected": {"ok": True, "data": {"value": 8}}},
    {"name": "negative", "input": {"value": -3},
     "expected": {"ok": True, "data": {"value": -2}}},
)
evaluation_examples = []
for backend in BACKENDS:
    application = examples[("System", "graph", backend)]
    evaluation = Eval(name="decision_contract", unit=application, cases=cases)
    evaluation_result = evaluation.run()

    report = evaluation_result.data
    assert evaluation_result.ok and report["total"] == report["passed"] == 2
    assert report["failed"] == 0
    for case, child in zip(report["cases"], evaluation_result.children):
        assert case["ok"] and case["result"]["execution_id"] == child.execution_id
        render_io(backend, {"input": case["input"], "expected": case["expected"]},
                  observed_process(child) + " → comparación exacta",
                  {"actual": child.data, "passed": case["ok"]})
    print(backend, {key: report[key] for key in ("total", "passed", "failed")})
    evaluation_examples.append({
        "backend": backend, "input": list(cases), "result": evaluation_result.to_dict(),
    })

Backend,Input,Proceso observado,Output,Estado
native,"{'input': {'value': 3}, 'expected': {'ok': True, 'data': {'value': 8}}}",increase: {'value': 4}; multiply: {'value': 8}; increase → multiply; multiply → END → comparación exacta,"{'actual': {'value': 8}, 'passed': True}",PASS


Backend,Input,Proceso observado,Output,Estado
native,"{'input': {'value': -3}, 'expected': {'ok': True, 'data': {'value': -2}}}",increase: {'value': -2}; increase → END → comparación exacta,"{'actual': {'value': -2}, 'passed': True}",PASS


native {'total': 2, 'passed': 2, 'failed': 0}


Backend,Input,Proceso observado,Output,Estado
langgraph,"{'input': {'value': 3}, 'expected': {'ok': True, 'data': {'value': 8}}}",increase: {'value': 4}; multiply: {'value': 8}; increase → multiply; multiply → END → comparación exacta,"{'actual': {'value': 8}, 'passed': True}",PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,"{'input': {'value': -3}, 'expected': {'ok': True, 'data': {'value': -2}}}",increase: {'value': -2}; increase → END → comparación exacta,"{'actual': {'value': -2}, 'passed': True}",PASS


langgraph {'total': 2, 'passed': 2, 'failed': 0}


### Comprobación: Eval detecta una expectativa incorrecta

Pedimos deliberadamente `999` cuando el System devuelve `8`. La ejecución del
System es correcta y **Eval debe fallar** al comparar. Esto comprueba que el
evaluador mide la salida y no se limita a comprobar que la ejecución terminó.

In [15]:
negative_controls = []
for backend in BACKENDS:
    application = examples[("System", "graph", backend)]
    wrong_cases = ({"name": "wrong_expected", "input": {"value": 3},
                    "expected": {"ok": True, "data": {"value": 999}}},)
    control = Eval(name="wrong_expectation", unit=application, cases=wrong_cases).run()
    assert not control.ok and control.data["failed"] == 1
    assert control.children[0].ok and control.children[0].data == {"value": 8}
    render_io(backend, wrong_cases[0], "System devuelve 8 → comparar con 999",
              {"actual": control.children[0].data, "eval_ok": control.ok,
               "failed": control.data["failed"]}, status="FALLO ESPERADO: detectado")
    negative_controls.append({"backend": backend, "detected": True, "result": control.to_dict()})

Backend,Input,Proceso observado,Output,Estado
native,"{'name': 'wrong_expected', 'input': {'value': 3}, 'expected': {'ok': True, 'data': {'value': 999}}}",System devuelve 8 → comparar con 999,"{'actual': {'value': 8}, 'eval_ok': False, 'failed': 1}",FALLO ESPERADO: detectado


Backend,Input,Proceso observado,Output,Estado
langgraph,"{'name': 'wrong_expected', 'input': {'value': 3}, 'expected': {'ok': True, 'data': {'value': 999}}}",System devuelve 8 → comparar con 999,"{'actual': {'value': 8}, 'eval_ok': False, 'failed': 1}",FALLO ESPERADO: detectado


## Verificación: las mismas conexiones sobre los seis ejemplos

No hace falta otra forma de construir Environment o Eval según el flujo. Ahora
reutilizamos las **12 unidades existentes** (seis ejemplos × dos backends).
Para cada una comprobamos dos registros de Environment y dos casos de Eval:

| Flujo | Input 3 | Input −3 |
|---|---:|---:|
| Secuencia | 8 | −4 |
| Paralelo | 10 | −8 |
| Decisión | 8 | −2 |

Esta celda añade comprobaciones de serialización, linaje, memoria e identidad de
los Agents. El código de presentación y verificación no es parte de la gramática.

In [16]:
def check_result_tree(result):
    nodes = list(result.walk())
    assert len({node.execution_id for node in nodes}) == len(nodes)
    for node in nodes:
        for child in node.children:
            assert child.parent_execution_id == node.execution_id
    toolkit.RunResult.model_validate(json.loads(json.dumps(result.to_dict(), allow_nan=False)))

expected_outputs = {"sequential": (8, -4), "parallel": (10, -8), "graph": (8, -2)}
boundary_matrix = []
for (owner, flow, backend), unit in examples.items():
    expected = [{"value": value} for value in expected_outputs[flow]]
    episode_result = Environment(name="coverage_episode", unit=unit, records=inputs).run()
    measured_cases = tuple(
        {"name": f"case_{index}", "input": payload,
         "expected": {"ok": True, "data": output}}
        for index, (payload, output) in enumerate(zip(inputs, expected))
    )
    evaluation_result = Eval(name="coverage_eval", unit=unit, cases=measured_cases).run()
    assert episode_result.ok and episode_result.data["outputs"] == expected
    assert episode_result.data["summary"]["memory"] == {"processed": 2}
    assert len(episode_result.data["history"]) == len(episode_result.children) == 2
    assert len({step["episode_id"] for step in episode_result.data["history"]}) == 1
    assert evaluation_result.ok and evaluation_result.data["passed"] == 2
    assert evaluation_result.data["failed"] == 0
    for result in (episode_result, evaluation_result):
        check_result_tree(result)
        assert len(result.children) == 2
        for child in result.children:
            agents = [n for n in child.walk() if n.meta.get("prototype_kind") == "agent"]
            assert agents and all(n.meta["skills"] == ["arithmetic"] for n in agents)
            assert all(event.name in {"increment", "double"}
                       for n in child.walk() for event in n.tool_events)
    boundary_matrix.append({
        "owner": owner, "flow": flow, "backend": backend,
        "environment_passed": True, "eval_passed": True,
        "inputs": list(inputs), "expected_outputs": expected,
        "environment": episode_result.to_dict(), "evaluation": evaluation_result.to_dict(),
    })
    print(f"{owner:6} {flow:10} {backend:9} → Environment: {expected}; Eval: 2/2 PASS")

assert len(boundary_matrix) == 12
assert len(negative_controls) == 2 and all(r["detected"] for r in negative_controls)
print("Environment: 12/12 unidades, 24/24 pasos; Eval: 12/12 unidades, 24/24 casos")

Agent  sequential native    → Environment: [{'value': 8}, {'value': -4}]; Eval: 2/2 PASS
Agent  sequential langgraph → Environment: [{'value': 8}, {'value': -4}]; Eval: 2/2 PASS
Agent  parallel   native    → Environment: [{'value': 10}, {'value': -8}]; Eval: 2/2 PASS
Agent  parallel   langgraph → Environment: [{'value': 10}, {'value': -8}]; Eval: 2/2 PASS
Agent  graph      native    → Environment: [{'value': 8}, {'value': -2}]; Eval: 2/2 PASS
Agent  graph      langgraph → Environment: [{'value': 8}, {'value': -2}]; Eval: 2/2 PASS


System sequential native    → Environment: [{'value': 8}, {'value': -4}]; Eval: 2/2 PASS
System sequential langgraph → Environment: [{'value': 8}, {'value': -4}]; Eval: 2/2 PASS


System parallel   native    → Environment: [{'value': 10}, {'value': -8}]; Eval: 2/2 PASS


System parallel   langgraph → Environment: [{'value': 10}, {'value': -8}]; Eval: 2/2 PASS


System graph      native    → Environment: [{'value': 8}, {'value': -2}]; Eval: 2/2 PASS
System graph      langgraph → Environment: [{'value': 8}, {'value': -2}]; Eval: 2/2 PASS
Environment: 12/12 unidades, 24/24 pasos; Eval: 12/12 unidades, 24/24 casos


## 10. Challenge realista — candidata 2.1.3 de revisión de API

Gestionamos pedidos de equipos con catálogo e inventario **locales y sintéticos**.
Input: identificador, país, moneda, presupuesto en centavos y líneas SKU/cantidad.
Output: decisión, importe, motivos y siguiente acción propuesta.

El System coordina **7 Agents y 7 Skills, con 8 Tools**:

1. **Intake:** normalizar y validar, en secuencia dentro de su Agent.
2. **Pricing e Inventory:** calcular precio y disponibilidad en paralelo.
3. **Policy:** aplicar presupuesto, stock y umbral de revisión.
4. **Approve, Review o Reject:** emitir la respuesta según la decisión.

Las conexiones condicionales son funciones directas. Las instrucciones finales
no envían mensajes, reservan stock ni crean pedidos reales. Las cantidades de
líneas repetidas del mismo SKU se suman antes de consultar el stock.

Esta es la **candidata de diseño de API 2.1.3**; el baseline ejecutado sigue siendo
el wheel publicado 2.1.2 con la fachada experimental. No se ha publicado 2.1.3.

In [17]:
import inspect
from IPython.display import Markdown
from unified_grammar.order_challenge import build_system, load_cases, verify

# Única declaración: notebook, tests y proyecto generado usan este constructor.
display(Markdown("```python\n" + inspect.getsource(build_system) + "\n```"))

```python
def build_system(backend="native"):
    intake = Agent("intake", Sequence((
        Action("normalize", "normalize_order", input=as_state),
        Action("validate", "validate_order", input=as_state),
    )), skills=(skills.intake,), backend=backend)
    pricing = Agent("pricing", Action("price", "price_order", input=as_state),
                    skills=(skills.pricing,), backend=backend)
    inventory = Agent("inventory", Action("stock", "check_inventory", input=as_state),
                      skills=(skills.inventory,), backend=backend)
    policy = Agent("policy", Action("assess", "assess_policy", input=as_state),
                   skills=(skills.policy,), backend=backend)
    approve = Agent("approve", Action("approve", "approve_order", input=as_state),
                    skills=(skills.approval,), backend=backend)
    review = Agent("review", Action("review", "review_order", input=as_state),
                   skills=(skills.review,), backend=backend)
    reject = Agent("reject", Action("reject", "reject_order", input=as_state),
                   skills=(skills.rejection,), backend=backend)
    return System("order_fulfillment", Graph(
        start="intake",
        nodes=(("intake", intake),
               ("checks", Parallel((pricing, inventory), join=merge_checks)),
               ("policy", policy), ("approve", approve),
               ("review", review), ("reject", reject)),
        edges=(("intake", after_intake), ("checks", "policy"),
               ("policy", after_policy), ("approve", "END"),
               ("review", "END"), ("reject", "END")),
    ), backend=backend)

```

### Input → proceso observado → output

Los siete casos comprueban aprobación, presupuesto insuficiente, falta de stock,
revisión por importe, cantidad inválida, SKU repetidos y un fallo técnico de
inventario. El rechazo de negocio es una ejecución correcta (`ok=True`). El
fallo técnico debe detener el flujo (`ok=False`) y conservar el precio calculado.

`verify` es el comprobador de contratos del ejemplo, no otro constructor de
gramática. Lee los contratos versionados y conserva eventos, rutas y linaje.
Las ramas paralelas se muestran en orden declarado, no como cronología.

In [18]:
challenge_reports = []
for backend in BACKENDS:
    report = verify(backend)
    challenge_reports.append(report)
    for case in report["cases"]:
        process = " → ".join(step["step"] + (" [fallo]" if not step["ok"] else "")
                             for step in case["process"])
        shown = case["output"] if case["ok"] else {
            "ok": False,
            "completed_steps": [s["step"] for s in case["process"] if s["ok"]],
            "failed_steps": [s["step"] for s in case["process"] if not s["ok"]],
        }
        render_io(backend, case["input"], process, shown,
                  status="PASS" if case["ok"] else "FALLO CONTROLADO: conservado")

assert [c["output"] for c in challenge_reports[0]["cases"]] == [
    c["output"] for c in challenge_reports[1]["cases"]
]
print("Challenge: 14/14 contratos (7 por backend), incluido el fallo técnico esperado")

Backend,Input,Proceso observado,Output,Estado
native,"{'order': {'id': 'PO-001', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 80000, 'items': [{'sku': 'KEYBOARD', 'quantity': 2}]}}",normalize → validate → price → stock → assess → approve,"{'order_id': 'PO-001', 'decision': 'approved', 'total_cents': 75000, 'currency': 'MXN', 'reasons': [], 'next_action': 'prepare_shipment'}",PASS


Backend,Input,Proceso observado,Output,Estado
native,"{'order': {'id': 'PO-002', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 60000, 'items': [{'sku': 'KEYBOARD', 'quantity': 2}]}}",normalize → validate → price → stock → assess → reject,"{'order_id': 'PO-002', 'decision': 'rejected', 'total_cents': 75000, 'currency': 'MXN', 'reasons': ['over_budget'], 'next_action': 'notify_customer'}",PASS


Backend,Input,Proceso observado,Output,Estado
native,"{'order': {'id': 'PO-003', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 2000000, 'items': [{'sku': 'MONITOR', 'quantity': 4}]}}",normalize → validate → price → stock → assess → reject,"{'order_id': 'PO-003', 'decision': 'rejected', 'total_cents': 1005000, 'currency': 'MXN', 'reasons': ['insufficient_stock'], 'next_action': 'notify_customer'}",PASS


Backend,Input,Proceso observado,Output,Estado
native,"{'order': {'id': 'PO-004', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 300000, 'items': [{'sku': 'MONITOR', 'quantity': 1}]}}",normalize → validate → price → stock → assess → review,"{'order_id': 'PO-004', 'decision': 'review', 'total_cents': 255000, 'currency': 'MXN', 'reasons': ['high_value_order'], 'next_action': 'request_human_approval'}",PASS


Backend,Input,Proceso observado,Output,Estado
native,"{'order': {'id': 'PO-005', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 80000, 'items': [{'sku': 'KEYBOARD', 'quantity': 0}]}}",normalize → validate → reject,"{'order_id': 'PO-005', 'decision': 'rejected', 'total_cents': None, 'currency': 'MXN', 'reasons': ['invalid_quantity'], 'next_action': 'notify_customer'}",PASS


...

Backend,Input,Proceso observado,Output,Estado
langgraph,"{'order': {'id': 'PO-004', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 300000, 'items': [{'sku': 'MONITOR', 'quantity': 1}]}}",normalize → validate → price → stock → assess → review,"{'order_id': 'PO-004', 'decision': 'review', 'total_cents': 255000, 'currency': 'MXN', 'reasons': ['high_value_order'], 'next_action': 'request_human_approval'}",PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,"{'order': {'id': 'PO-005', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 80000, 'items': [{'sku': 'KEYBOARD', 'quantity': 0}]}}",normalize → validate → reject,"{'order_id': 'PO-005', 'decision': 'rejected', 'total_cents': None, 'currency': 'MXN', 'reasons': ['invalid_quantity'], 'next_action': 'notify_customer'}",PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,"{'order': {'id': 'PO-006', 'currency': 'MXN', 'country': 'US', 'budget_cents': 2000000, 'items': [{'sku': 'DOCK', 'quantity': 3}, {'sku': 'DOCK', 'quantity': 3}]}}",normalize → validate → price → stock → assess → reject,"{'order_id': 'PO-006', 'decision': 'rejected', 'total_cents': 732000, 'currency': 'MXN', 'reasons': ['insufficient_stock'], 'next_action': 'notify_customer'}",PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,"{'order': {'id': 'PO-007', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 80000, 'items': [{'sku': 'KEYBOARD', 'quantity': 2}]}, 'simulate_inventory_failure': True}",normalize → validate → price → stock [fallo],"{'ok': False, 'completed_steps': ['normalize', 'validate', 'price'], 'failed_steps': ['stock']}",FALLO CONTROLADO: conservado


Challenge: 14/14 contratos (7 por backend), incluido el fallo técnico esperado


### Environment y Eval sobre el challenge

Environment entrega los seis pedidos de negocio al mismo System, conserva cada
transición y cuenta los registros procesados. El stock es una fotografía fija
por pedido: no es un almacén con reservas persistentes entre pasos.

Eval vuelve a ejecutar esos seis pedidos y compara las respuestas exactas.
El fallo técnico se verifica por separado; no se presenta como una evaluación
positiva. Un control con un centavo de diferencia debe fallar.

In [19]:
business_cases = tuple(case for case in load_cases() if case["expected"]["ok"])
challenge_episodes = []
for backend in BACKENDS:
    application = build_system(backend)
    episode = Environment(name="orders", unit=application,
                          records=tuple(case["input"] for case in business_cases))
    episode_result = episode.run()
    evaluation = Eval(name="order_contract", unit=application, cases=business_cases)
    evaluation_result = evaluation.run()
    assert episode_result.ok and evaluation_result.ok
    assert episode_result.data["outputs"] == [case["expected"]["data"] for case in business_cases]
    assert episode_result.data["summary"]["memory"] == {"processed": 6}
    assert evaluation_result.data["passed"] == 6 and evaluation_result.data["failed"] == 0
    for case, child in zip(business_cases, episode_result.children):
        render_io(backend, case["input"], observed_process(child), child.data)
    negative = next(r for r in challenge_reports if r["backend"] == backend)["negative_control"]
    assert not negative["ok"] and negative["data"]["failed"] == 1
    print(backend, {"episode_steps": 6, "memory": episode_result.data["summary"]["memory"],
                    "eval_passed": evaluation_result.data["passed"], "wrong_cent_detected": True})
    challenge_episodes.append({"backend": backend, "environment": episode_result.to_dict(),
                               "evaluation": evaluation_result.to_dict()})

challenge_destination = source_root / "unified_grammar/evidence/challenge.json"
challenge_destination.parent.mkdir(exist_ok=True)
challenge_destination.write_text(json.dumps({
    "target_version": "2.1.3", "status": "offline-api-revision-candidate",
    "baseline_version": toolkit.__version__, "inference_requests": 0,
    "reports": challenge_reports, "demonstrated_episodes": challenge_episodes,
}, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Evidencia:", challenge_destination)

Backend,Input,Proceso observado,Output,Estado
native,"{'order': {'id': 'PO-001', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 80000, 'items': [{'sku': 'KEYBOARD', 'quantity': 2}]}}","normalize: {'order': {'id': 'PO-001', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 80000, 'items': [{'sku': 'KEYBOARD', 'quantity': 2}]}}; validate: {'order': {'id': 'PO-001', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 80000, 'items': [{'sku': 'KEYBOARD', 'quantity': 2}]}, 'validation': {'valid': True, 'issues': []}}; price: {'order': {'id': 'PO-001', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 80000, 'items': [{'sku': 'KEYBOARD', 'quantity': 2}]}, 'validation': {'valid': True, 'issues': []}, 'pricing': {'subtotal_cents': 70000, 'shipping_cents': 5000, 'total_cents': 75000}}; stock: {'order': {'id': 'PO-001', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 80000, 'items': [{'sku': 'KEYBOARD', 'quantity': 2}]}, 'validation': {'valid': True, 'issues': []}, 'inventory': {'available': True, 'shortages': [], 'requested': {'KEYBOARD': 2}}}; assess: {'order': {'id': 'PO-001', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 80000, 'items': [{'sku': 'KEYBOARD', 'quantity': 2}]}, 'validation': {'valid': True, 'issues': []}, 'pricing': {'subtotal_cents': 70000, 'shipping_cents': 5000, 'total_cents': 75000}, 'inventory': {'available': True, 'shortages': [], 'requested': {'KEYBOARD': 2}}, 'policy': {'decision': 'approved', 'reasons': []}}; approve: {'order_id': 'PO-001', 'decision': 'approved', 'total_cents': 75000, 'currency': 'MXN', 'reasons': [], 'next_action': 'prepare_shipment'}; intake → checks; checks → policy; policy → approve; approve → END","{'order_id': 'PO-001', 'decision': 'approved', 'total_cents': 75000, 'currency': 'MXN', 'reasons': [], 'next_action': 'prepare_shipment'}",PASS


Backend,Input,Proceso observado,Output,Estado
native,"{'order': {'id': 'PO-002', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 60000, 'items': [{'sku': 'KEYBOARD', 'quantity': 2}]}}","normalize: {'order': {'id': 'PO-002', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 60000, 'items': [{'sku': 'KEYBOARD', 'quantity': 2}]}}; validate: {'order': {'id': 'PO-002', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 60000, 'items': [{'sku': 'KEYBOARD', 'quantity': 2}]}, 'validation': {'valid': True, 'issues': []}}; price: {'order': {'id': 'PO-002', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 60000, 'items': [{'sku': 'KEYBOARD', 'quantity': 2}]}, 'validation': {'valid': True, 'issues': []}, 'pricing': {'subtotal_cents': 70000, 'shipping_cents': 5000, 'total_cents': 75000}}; stock: {'order': {'id': 'PO-002', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 60000, 'items': [{'sku': 'KEYBOARD', 'quantity': 2}]}, 'validation': {'valid': True, 'issues': []}, 'inventory': {'available': True, 'shortages': [], 'requested': {'KEYBOARD': 2}}}; assess: {'order': {'id': 'PO-002', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 60000, 'items': [{'sku': 'KEYBOARD', 'quantity': 2}]}, 'validation': {'valid': True, 'issues': []}, 'pricing': {'subtotal_cents': 70000, 'shipping_cents': 5000, 'total_cents': 75000}, 'inventory': {'available': True, 'shortages': [], 'requested': {'KEYBOARD': 2}}, 'policy': {'decision': 'rejected', 'reasons': ['over_budget']}}; reject: {'order_id': 'PO-002', 'decision': 'rejected', 'total_cents': 75000, 'currency': 'MXN', 'reasons': ['over_budget'], 'next_action': 'notify_customer'}; intake → checks; checks → policy; policy → reject; reject → END","{'order_id': 'PO-002', 'decision': 'rejected', 'total_cents': 75000, 'currency': 'MXN', 'reasons': ['over_budget'], 'next_action': 'notify_customer'}",PASS


Backend,Input,Proceso observado,Output,Estado
native,"{'order': {'id': 'PO-003', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 2000000, 'items': [{'sku': 'MONITOR', 'quantity': 4}]}}","normalize: {'order': {'id': 'PO-003', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 2000000, 'items': [{'sku': 'MONITOR', 'quantity': 4}]}}; validate: {'order': {'id': 'PO-003', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 2000000, 'items': [{'sku': 'MONITOR', 'quantity': 4}]}, 'validation': {'valid': True, 'issues': []}}; price: {'order': {'id': 'PO-003', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 2000000, 'items': [{'sku': 'MONITOR', 'quantity': 4}]}, 'validation': {'valid': True, 'issues': []}, 'pricing': {'subtotal_cents': 1000000, 'shipping_cents': 5000, 'total_cents': 1005000}}; stock: {'order': {'id': 'PO-003', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 2000000, 'items': [{'sku': 'MONITOR', 'quantity': 4}]}, 'validation': {'valid': True, 'issues': []}, 'inventory': {'available': False, 'shortages': ['MONITOR'], 'requested': {'MONITOR': 4}}}; assess: {'order': {'id': 'PO-003', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 2000000, 'items': [{'sku': 'MONITOR', 'quantity': 4}]}, 'validation': {'valid': True, 'issues': []}, 'pricing': {'subtotal_cents': 1000000, 'shipping_cents': 5000, 'total_cents': 1005000}, 'inventory': {'available': False, 'shortages': ['MONITOR'], 'requested': {'MONITOR': 4}}, 'policy': {'decision': 'rejected', 'reasons': ['insufficient_stock']}}; reject: {'order_id': 'PO-003', 'decision': 'rejected', 'total_cents': 1005000, 'currency': 'MXN', 'reasons': ['insufficient_stock'], 'next_action': 'notify_customer'}; intake → checks; checks → policy; policy → reject; reject → END","{'order_id': 'PO-003', 'decision': 'rejected', 'total_cents': 1005000, 'currency': 'MXN', 'reasons': ['insufficient_stock'], 'next_action': 'notify_customer'}",PASS


Backend,Input,Proceso observado,Output,Estado
native,"{'order': {'id': 'PO-004', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 300000, 'items': [{'sku': 'MONITOR', 'quantity': 1}]}}","normalize: {'order': {'id': 'PO-004', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 300000, 'items': [{'sku': 'MONITOR', 'quantity': 1}]}}; validate: {'order': {'id': 'PO-004', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 300000, 'items': [{'sku': 'MONITOR', 'quantity': 1}]}, 'validation': {'valid': True, 'issues': []}}; price: {'order': {'id': 'PO-004', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 300000, 'items': [{'sku': 'MONITOR', 'quantity': 1}]}, 'validation': {'valid': True, 'issues': []}, 'pricing': {'subtotal_cents': 250000, 'shipping_cents': 5000, 'total_cents': 255000}}; stock: {'order': {'id': 'PO-004', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 300000, 'items': [{'sku': 'MONITOR', 'quantity': 1}]}, 'validation': {'valid': True, 'issues': []}, 'inventory': {'available': True, 'shortages': [], 'requested': {'MONITOR': 1}}}; assess: {'order': {'id': 'PO-004', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 300000, 'items': [{'sku': 'MONITOR', 'quantity': 1}]}, 'validation': {'valid': True, 'issues': []}, 'pricing': {'subtotal_cents': 250000, 'shipping_cents': 5000, 'total_cents': 255000}, 'inventory': {'available': True, 'shortages': [], 'requested': {'MONITOR': 1}}, 'policy': {'decision': 'review', 'reasons': ['high_value_order']}}; review: {'order_id': 'PO-004', 'decision': 'review', 'total_cents': 255000, 'currency': 'MXN', 'reasons': ['high_value_order'], 'next_action': 'request_human_approval'}; intake → checks; checks → policy; policy → review; review → END","{'order_id': 'PO-004', 'decision': 'review', 'total_cents': 255000, 'currency': 'MXN', 'reasons': ['high_value_order'], 'next_action': 'request_human_approval'}",PASS


Backend,Input,Proceso observado,Output,Estado
native,"{'order': {'id': 'PO-005', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 80000, 'items': [{'sku': 'KEYBOARD', 'quantity': 0}]}}","normalize: {'order': {'id': 'PO-005', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 80000, 'items': [{'sku': 'KEYBOARD', 'quantity': 0}]}}; validate: {'order': {'id': 'PO-005', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 80000, 'items': [{'sku': 'KEYBOARD', 'quantity': 0}]}, 'validation': {'valid': False, 'issues': ['invalid_quantity']}}; reject: {'order_id': 'PO-005', 'decision': 'rejected', 'total_cents': None, 'currency': 'MXN', 'reasons': ['invalid_quantity'], 'next_action': 'notify_customer'}; intake → reject; reject → END","{'order_id': 'PO-005', 'decision': 'rejected', 'total_cents': None, 'currency': 'MXN', 'reasons': ['invalid_quantity'], 'next_action': 'notify_customer'}",PASS


...

Backend,Input,Proceso observado,Output,Estado
langgraph,"{'order': {'id': 'PO-003', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 2000000, 'items': [{'sku': 'MONITOR', 'quantity': 4}]}}","normalize: {'order': {'id': 'PO-003', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 2000000, 'items': [{'sku': 'MONITOR', 'quantity': 4}]}}; validate: {'order': {'id': 'PO-003', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 2000000, 'items': [{'sku': 'MONITOR', 'quantity': 4}]}, 'validation': {'valid': True, 'issues': []}}; price: {'order': {'id': 'PO-003', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 2000000, 'items': [{'sku': 'MONITOR', 'quantity': 4}]}, 'validation': {'valid': True, 'issues': []}, 'pricing': {'subtotal_cents': 1000000, 'shipping_cents': 5000, 'total_cents': 1005000}}; stock: {'order': {'id': 'PO-003', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 2000000, 'items': [{'sku': 'MONITOR', 'quantity': 4}]}, 'validation': {'valid': True, 'issues': []}, 'inventory': {'available': False, 'shortages': ['MONITOR'], 'requested': {'MONITOR': 4}}}; assess: {'order': {'id': 'PO-003', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 2000000, 'items': [{'sku': 'MONITOR', 'quantity': 4}]}, 'validation': {'valid': True, 'issues': []}, 'pricing': {'subtotal_cents': 1000000, 'shipping_cents': 5000, 'total_cents': 1005000}, 'inventory': {'available': False, 'shortages': ['MONITOR'], 'requested': {'MONITOR': 4}}, 'policy': {'decision': 'rejected', 'reasons': ['insufficient_stock']}}; reject: {'order_id': 'PO-003', 'decision': 'rejected', 'total_cents': 1005000, 'currency': 'MXN', 'reasons': ['insufficient_stock'], 'next_action': 'notify_customer'}; intake → checks; checks → policy; policy → reject; reject → END","{'order_id': 'PO-003', 'decision': 'rejected', 'total_cents': 1005000, 'currency': 'MXN', 'reasons': ['insufficient_stock'], 'next_action': 'notify_customer'}",PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,"{'order': {'id': 'PO-004', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 300000, 'items': [{'sku': 'MONITOR', 'quantity': 1}]}}","normalize: {'order': {'id': 'PO-004', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 300000, 'items': [{'sku': 'MONITOR', 'quantity': 1}]}}; validate: {'order': {'id': 'PO-004', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 300000, 'items': [{'sku': 'MONITOR', 'quantity': 1}]}, 'validation': {'valid': True, 'issues': []}}; price: {'order': {'id': 'PO-004', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 300000, 'items': [{'sku': 'MONITOR', 'quantity': 1}]}, 'validation': {'valid': True, 'issues': []}, 'pricing': {'subtotal_cents': 250000, 'shipping_cents': 5000, 'total_cents': 255000}}; stock: {'order': {'id': 'PO-004', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 300000, 'items': [{'sku': 'MONITOR', 'quantity': 1}]}, 'validation': {'valid': True, 'issues': []}, 'inventory': {'available': True, 'shortages': [], 'requested': {'MONITOR': 1}}}; assess: {'order': {'id': 'PO-004', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 300000, 'items': [{'sku': 'MONITOR', 'quantity': 1}]}, 'validation': {'valid': True, 'issues': []}, 'pricing': {'subtotal_cents': 250000, 'shipping_cents': 5000, 'total_cents': 255000}, 'inventory': {'available': True, 'shortages': [], 'requested': {'MONITOR': 1}}, 'policy': {'decision': 'review', 'reasons': ['high_value_order']}}; review: {'order_id': 'PO-004', 'decision': 'review', 'total_cents': 255000, 'currency': 'MXN', 'reasons': ['high_value_order'], 'next_action': 'request_human_approval'}; intake → checks; checks → policy; policy → review; review → END","{'order_id': 'PO-004', 'decision': 'review', 'total_cents': 255000, 'currency': 'MXN', 'reasons': ['high_value_order'], 'next_action': 'request_human_approval'}",PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,"{'order': {'id': 'PO-005', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 80000, 'items': [{'sku': 'KEYBOARD', 'quantity': 0}]}}","normalize: {'order': {'id': 'PO-005', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 80000, 'items': [{'sku': 'KEYBOARD', 'quantity': 0}]}}; validate: {'order': {'id': 'PO-005', 'currency': 'MXN', 'country': 'MX', 'budget_cents': 80000, 'items': [{'sku': 'KEYBOARD', 'quantity': 0}]}, 'validation': {'valid': False, 'issues': ['invalid_quantity']}}; reject: {'order_id': 'PO-005', 'decision': 'rejected', 'total_cents': None, 'currency': 'MXN', 'reasons': ['invalid_quantity'], 'next_action': 'notify_customer'}; intake → reject; reject → END","{'order_id': 'PO-005', 'decision': 'rejected', 'total_cents': None, 'currency': 'MXN', 'reasons': ['invalid_quantity'], 'next_action': 'notify_customer'}",PASS


Backend,Input,Proceso observado,Output,Estado
langgraph,"{'order': {'id': 'PO-006', 'currency': 'MXN', 'country': 'US', 'budget_cents': 2000000, 'items': [{'sku': 'DOCK', 'quantity': 3}, {'sku': 'DOCK', 'quantity': 3}]}}","normalize: {'order': {'id': 'PO-006', 'currency': 'MXN', 'country': 'US', 'budget_cents': 2000000, 'items': [{'sku': 'DOCK', 'quantity': 3}, {'sku': 'DOCK', 'quantity': 3}]}}; validate: {'order': {'id': 'PO-006', 'currency': 'MXN', 'country': 'US', 'budget_cents': 2000000, 'items': [{'sku': 'DOCK', 'quantity': 3}, {'sku': 'DOCK', 'quantity': 3}]}, 'validation': {'valid': True, 'issues': []}}; price: {'order': {'id': 'PO-006', 'currency': 'MXN', 'country': 'US', 'budget_cents': 2000000, 'items': [{'sku': 'DOCK', 'quantity': 3}, {'sku': 'DOCK', 'quantity': 3}]}, 'validation': {'valid': True, 'issues': []}, 'pricing': {'subtotal_cents': 720000, 'shipping_cents': 12000, 'total_cents': 732000}}; stock: {'order': {'id': 'PO-006', 'currency': 'MXN', 'country': 'US', 'budget_cents': 2000000, 'items': [{'sku': 'DOCK', 'quantity': 3}, {'sku': 'DOCK', 'quantity': 3}]}, 'validation': {'valid': True, 'issues': []}, 'inventory': {'available': False, 'shortages': ['DOCK'], 'requested': {'DOCK': 6}}}; assess: {'order': {'id': 'PO-006', 'currency': 'MXN', 'country': 'US', 'budget_cents': 2000000, 'items': [{'sku': 'DOCK', 'quantity': 3}, {'sku': 'DOCK', 'quantity': 3}]}, 'validation': {'valid': True, 'issues': []}, 'pricing': {'subtotal_cents': 720000, 'shipping_cents': 12000, 'total_cents': 732000}, 'inventory': {'available': False, 'shortages': ['DOCK'], 'requested': {'DOCK': 6}}, 'policy': {'decision': 'rejected', 'reasons': ['insufficient_stock']}}; reject: {'order_id': 'PO-006', 'decision': 'rejected', 'total_cents': 732000, 'currency': 'MXN', 'reasons': ['insufficient_stock'], 'next_action': 'notify_customer'}; intake → checks; checks → policy; policy → reject; reject → END","{'order_id': 'PO-006', 'decision': 'rejected', 'total_cents': 732000, 'currency': 'MXN', 'reasons': ['insufficient_stock'], 'next_action': 'notify_customer'}",PASS


langgraph {'episode_steps': 6, 'memory': {'processed': 6}, 'eval_passed': 6, 'wrong_cent_detected': True}
Evidencia: C:\Users\jacob\Documents\AVANTECK.TEAM\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\examples\unified_grammar\evidence\challenge.json


### Scaffolder de la revisión 2.1.3

`python examples/unified_grammar/scaffold.py /ruta/a/proyecto_nuevo`
genera un proyecto independiente con `app.py`, `src/order_challenge/`, el módulo
experimental `src/unified_grammar/`, fixtures, notebook, tests, configuración,
manifiesto y checksums. Rechaza destinos existentes y no modifica el SDK.

El notebook del proyecto generado importa este mismo `build_system`; no copia
su lógica. Consulta `docs/SCAFFOLDER_213.md` en el repositorio para el árbol
completo de la librería y la ubicación de cada responsabilidad.

## Preparación opcional y evidencia

El uso habitual fue `run(input)`. Ahora verificamos que preparar explícitamente
con `compile()` conserva los resultados y que LangGraph contiene un
`CompiledStateGraph` real. También contrastamos estos ejemplos con `demo.py`.
Esta sección es comprobación del prototipo, no otra gramática que aprender.

In [20]:
from unified_grammar.demo import build

for (unit, payload, expected), row in zip(prepared_units, records):
    prepared = unit.compile()
    assert prepared.inspect()["backend"] == row["backend"]
    if row["backend"] == "langgraph":
        assert type(prepared.prepared.children[0].native).__name__ == "CompiledStateGraph"
    result = prepared.run(payload)
    assert result.ok and result.data == expected
    reference = build(row["owner"], row["flow"], row["backend"]).run(payload)
    assert reference.ok and reference.data == result.data

assert len(toolkit.__all__) == 89
assert blocked_attempts() == 0
evidence = {
    "schema_version": "unified-grammar.six-examples.v1",
    "python_version": sys.version.split()[0],
    "package_version": toolkit.__version__,
    "package_file": str(Path(toolkit.__file__).resolve()),
    "langgraph_version": metadata.version("langgraph"),
    "inference_requests": 0,
    "blocked_network_or_subprocess_attempts": blocked_attempts(),
    "principal_passed": len(principal),
    "alternatives_passed": len(alternatives),
    "compiled_checks_passed": len(prepared_units),
    "shared_demo_checks_passed": len(records),
    "records": records,
    "skill_connections": skill_connections,
    "environment_examples": environment_examples,
    "evaluation_examples": evaluation_examples,
    "negative_controls": negative_controls,
    "boundary_matrix": boundary_matrix,
    "environment_units_passed": sum(row["environment_passed"] for row in boundary_matrix),
    "eval_units_passed": sum(row["eval_passed"] for row in boundary_matrix),
    "environment_steps_passed": sum(len(row["environment"]["children"]) for row in boundary_matrix),
    "eval_cases_passed": sum(row["evaluation"]["data"]["passed"] for row in boundary_matrix),
    "negative_controls_detected": len(negative_controls),
    "target_api_revision": "2.1.3",
    "challenge_evidence": "challenge.json",
    "challenge_contracts_passed": sum(r["cases_passed"] for r in challenge_reports),
}
destination = source_root / "unified_grammar/evidence/six-examples.json"
destination.write_text(json.dumps(evidence, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print({"matriz": "12/12", "alternativas": "4/4", "preparadas": "16/16",
       "environment": "12/12 unidades; 24/24 pasos", "eval": "12/12 unidades; 24/24 casos",
       "controles_negativos": "2/2 detectados",
       "inferencia": 0, "evidencia": str(destination)})

{'matriz': '12/12', 'alternativas': '4/4', 'preparadas': '16/16', 'environment': '12/12 unidades; 24/24 pasos', 'eval': '12/12 unidades; 24/24 casos', 'controles_negativos': '2/2 detectados', 'inferencia': 0, 'evidencia': 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\examples\\unified_grammar\\evidence\\six-examples.json'}


## Alcance

El paquete publicado 2.1.2 y sus 89 exports permanecen intactos. La nueva sintaxis
se prepara como candidata de revisión de API 2.1.3 y vive en este ejemplo
experimental. `Route` sigue siendo aceptado
para compatibilidad con el prototipo anterior; estos ejemplos enseñan funciones
directas. Las pruebas externas cubren además destinos inválidos, excepciones,
ciclos limitados y compilación sin ejecutar condiciones.

No se prueba razonamiento de modelos ni proveedores. Los límites son cooperativos;
el guard es del proceso Python y no constituye un sandbox del sistema operativo.
Skills, Environment y Eval se han conectado aquí a las mismas unidades de los
seis ejemplos. Environment y Eval no incorporan su propio selector de backend:
usan la configuración de su `unit`. Los wrappers son síncronos y no implementan
evaluación de Environment directamente; Eval recibe un Agent o System.